# Projekt 6 — Problem wieloagentowy: magazyn z 3 robotami

**Autorzy:** Konrad Małek, Barłomiej Mazgaj  
**Przedmiot:** Inteligencja obliczeniowa

## Spis treści
1. Opis zadania i środowiska
2. Reprezentacja stanu i akcji
3. Algorytmy i ich warianty
4. Trening homogeniczny — krzywe uczenia
5. Trening heterogeniczny (różne algorytmy w jednym epizodzie)
6. Porównanie i dyskusja
7. Wnioski


In [ ]:
import json
import os
import sys
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path('..').resolve()  # warehouse/
sys.path.insert(0, str(ROOT))

LOGS_DIR = ROOT / 'logs'
MODELS_DIR = ROOT / 'models'
EVAL_DIR = ROOT / 'report' / 'eval'
GIFS_DIR = ROOT / 'report' / 'gifs'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
GIFS_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)

## 1. Opis zadania i środowiska

Środowisko `WarehouseMultiEnv` (zaimplementowane w `warehouse/env/warehouse_multi_env.py`) jest rozszerzeniem naszego wcześniejszego, jednorobotowego środowiska z Projektu 5 do scenariusza wieloagentowego. Mapa to kwadrat **14 × 14 m**, w którym ustawiono 6 prostokątnych półek formujących trzy kolumny — pomiędzy kolumnami zostawiono korytarze szerokie na 2 m, co wymusza na agentach planowanie omijania zarówno przeszkód statycznych, jak i siebie nawzajem. Na mapie znajduje się **6 paczek** do zebrania oraz jedna baza dostaw w prawym górnym rogu.

Każdy z **3 robotów** startuje w innym rogu i porusza się niezależnie. Roboty są kooperatywne: dzielą **wspólną nagrodę zespołową** (shared reward), co matematycznie ujmuje to, że celem jest jak najszybsze dostarczenie wszystkich paczek niezależnie od tego, który robot je dostarczy. Kolizje:
- robot↔półka / robot↔ściana mapy: kara −5,
- robot↔robot: kara −3 (każdy z dwóch zderzających się dostaje karę),
- agent po kolizji nie zmienia położenia w danym kroku (prędkość zerowana).

Sensory ray-castingowe każdego robota wykrywają **zarówno statyczne przeszkody, jak i innych robotów**, dzięki czemu polityki uczą się unikać kolizji z dynamicznymi obiektami.

In [ ]:
# Wizualizacja layoutu środowiska (1 klatka po reset)
from env.warehouse_multi_env import WarehouseMultiEnv
env_viz = WarehouseMultiEnv(n_agents=3, n_packages=6, render_mode='rgb_array')
env_viz.reset(seed=0)
frame = env_viz.render()
env_viz.close()
plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.axis('off')
plt.title('WarehouseMultiEnv — layout początkowy (3 roboty, 6 paczek, 6 półek)')
plt.tight_layout()
plt.show()

## 2. Reprezentacja stanu i akcji

**Przestrzeń akcji** (per agent): `Box(-1, 1, shape=(2,))` — znormalizowane przyspieszenie `(a_x, a_y)`. Akcja jest mnożona przez `A_MAX = 1.0` i całkowana do prędkości w czasie `DT = 0.1 s`, z saturacją `|v| ≤ V_MAX = 2 m/s`.

**Przestrzeń obserwacji** (per agent, dim = **23**): wszystkie składowe znormalizowane do `[-1, 1]`:

| indeks | składowa | opis |
|---|---|---|
| 0–1 | pozycja własna | `(x, y) / map_size · 2 − 1` |
| 2–3 | prędkość własna | `v / V_MAX` |
| 4 | flaga carrying | 1 jeśli niesie paczkę |
| 5–7 | wektor do celu | (Δx, Δy, dystans) do najbliższej paczki lub bazy |
| 8–15 | 8 sensorów | odległość do przeszkody lub innego robota (1.0 = pusto) |
| 16 | % paczek pozostałych | `n_left / n_initial` |
| 17–22 | inni agenci | dla każdego z 2 pozostałych: `(rel_x, rel_y, carrying)` |

Każdy agent ma więc *lokalną* obserwację — wie gdzie są pozostali (relatywnie), ale nie zna ich aktualnego celu (musi się go domyślić po pozycji i flagach).

## 3. Algorytmy i ich warianty

Do porównania wybraliśmy z biblioteki **Stable Baselines 3** dwa algorytmy:

**PPO (Proximal Policy Optimization)** — on-policy, działa na rollout-buffer-ze. Stabilny, dobrze radzi sobie z ciągłymi akcjami, ale wymaga dużej liczby kroków. Porównujemy **dwa warianty hiperparametrów**:

| param | PPO-v1 (baseline) | PPO-v2 (wolniejszy uczeń, dłuższe rolloutsy) |
|---|---|---|
| `learning_rate` | 3e-4 | 1e-4 |
| `n_steps` | 2048 | 4096 |
| `batch_size` | 64 | 128 |
| `gamma` | 0.99 | 0.995 |
| `gae_lambda` | 0.95 | 0.98 |
| `ent_coef` | 0.0 | 0.01 |

**SAC (Soft Actor-Critic)** — off-policy, replay buffer + maksymalizacja entropii. Wymaga mniej kroków środowiska, ale każdy krok kosztuje więcej obliczeń (gradient step na każdy step). Konfiguracja:

- `learning_rate=3e-4`, `buffer_size=200k`, `batch_size=256`, `tau=0.005`, `gamma=0.99`, `train_freq=1`, `gradient_steps=1`, `learning_starts=5k`.

**Tryb homogeniczny**: jedna polityka (PPO-v1, PPO-v2 lub SAC) sterująca wszystkimi trzema robotami (parameter sharing przez SuperSuit `pettingzoo_env_to_vec_env_v1`).

**Tryb heterogeniczny**: każdy agent ma własną politykę z innym algorytmem (`robot_0` = PPO-v1, `robot_1` = PPO-v2, `robot_2` = SAC). Trening prowadzony **round-robin**: w jednej rundzie jeden agent się uczy, pozostali działają jako *zamrożone* polityki. Dzięki temu w trakcie każdego epizodu uczenia w środowisku **rzeczywiście współdziałają trzy różne algorytmy** — co pokrywa wymaganie polecenia o eksperymentach z różnymi algorytmami w jednym epizodzie.

In [ ]:
# ── Loader krzywych uczenia z plików tfevents ─────────────────────────
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def load_tb_scalars(run_dir: Path, tags: list[str]) -> dict[str, tuple[list[int], list[float]]]:
    """Zwraca {tag: (steps, values)} dla najlepszego (najnowszego) tfevents-a w run_dir."""
    candidates = sorted(glob(str(run_dir / '**' / 'events.out.tfevents*'), recursive=True))
    if not candidates:
        return {}
    ea = EventAccumulator(os.path.dirname(candidates[-1]), size_guidance={'scalars': 0})
    ea.Reload()
    out = {}
    for tag in tags:
        if tag in ea.Tags().get('scalars', []):
            evts = ea.Scalars(tag)
            out[tag] = ([e.step for e in evts], [e.value for e in evts])
    return out

RUNS_HOMO = {
    'PPO v1': 'homo_ppo_v1',
    'PPO v2': 'homo_ppo_v2',
    'SAC':    'homo_sac_v1',
}
RUN_HETERO = 'hetero_ppo-v1_ppo-v2_sac'

# Standardowe tagi SB3:
#  - 'rollout/ep_rew_mean'  (PPO + SAC)
#  - 'rollout/ep_len_mean'  (PPO + SAC)
#  - 'eval/mean_reward'     (z EvalCallback)
TAGS = ['rollout/ep_rew_mean', 'rollout/ep_len_mean', 'eval/mean_reward']

## 4. Trening homogeniczny — krzywe uczenia

Poniżej krzywe rolloutowej średniej nagrody (`rollout/ep_rew_mean`) z TensorBoard, dla każdego z trzech biegów homogenicznych. Pamiętaj: po włączeniu parameter-sharing przez SuperSuit `num_envs = n_agents × n_parallel_envs`, więc oś X (timesteps) zawiera w sobie wszystkie agent-steps.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
for label, run in RUNS_HOMO.items():
    data = load_tb_scalars(LOGS_DIR / run, TAGS)
    if 'eval/mean_reward' in data:
        steps, vals = data['eval/mean_reward']
        ax.plot(steps, vals, label=label, linewidth=1.6)
    else:
        print(f'[brak] eval/mean_reward dla {run} — czy bieg był odpalony?')
ax.set_xlabel('timesteps')
ax.set_ylabel('mean episode reward (eval)')
ax.set_title('Krzywe uczenia (eval) — tryb homogeniczny')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_DIR / 'learning_curves_homo.png', dpi=120)
plt.show()

In [ ]:
# Eval/mean_reward — wynik na deterministycznym EvalCallbacku co 20k kroków
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
for label, run in RUNS_HOMO.items():
    data = load_tb_scalars(LOGS_DIR / run, TAGS)
    if 'eval/mean_reward' in data:
        steps, vals = data['eval/mean_reward']
        ax.plot(steps, vals, label=label, marker='o', linewidth=1.6, markersize=4)
ax.set_xlabel('timesteps')
ax.set_ylabel('eval mean reward (deterministic)')
ax.set_title('Krzywe ewaluacji deterministycznej — homo')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_DIR / 'eval_curves_homo.png', dpi=120)
plt.show()

### 4.1 Metryki końcowe (30 epizodów, deterministycznie)

In [ ]:
rows = []
for label, run in RUNS_HOMO.items():
    p = EVAL_DIR / f'{run}.json'
    if p.exists():
        agg = json.loads(p.read_text())['agg']
        rows.append({'bieg': label, **agg})
if rows:
    df = pd.DataFrame(rows).set_index('bieg')
    display(df.round(2))
else:
    display(Markdown('_Brak plików JSON z metrykami — uruchom `eval_multi.py` zgodnie z instrukcją powyżej._'))

#### Obserwacje — tryb homogeniczny

- **Oba warianty PPO zbiegają niemal identycznie** (PPO-v1: **+163.9**, PPO-v2: **+162.7**). Krzywa `eval/mean_reward` w obu przypadkach startuje ~−5900 i dochodzi do ~+165 — bardziej konserwatywne ustawienia v2 (niższy `learning_rate`, dłuższe rolloutsy, dodatnia entropia) **nie dały przy tym budżecie kroków zauważalnie innego wyniku końcowego**. To samo w sobie jest wynikiem: dla tego zadania polityka jest mało wrażliwa na tę zmianę hiperparametrów.
- **PPO uczy się sensownej polityki**: dostarcza średnio **3 z 6** paczek przy zaledwie **1 kolizji** na epizod — czyli skutecznie omija półki i pozostałe roboty. Parameter sharing oznacza, że wszystkie trzy roboty dzielą tę samą, spójną politykę unikania.
- **SAC kolapsuje** (−214.4, **0 dostaw**, 0 kolizji). Krzywa eval rośnie z ~−3600 tylko do ~−240 i nigdy nie przebija zera. Polityka degeneruje się do „stój / minimalizuj ruch": zero kolizji, ale też zero zaangażowania w zbieranie paczek. Jest to spójne z wcześniejszą diagnozą, że to **skala kar** (a nie losowość) wybija SAC — przy dominujących karach lokalnie najbezpieczniejszą polityką jest bezruch.
- **`success_rate = 0` wszędzie** wynika z definicji sukcesu (wszystkie 6 paczek dostarczone w ≤ 800 krokach) oraz `steps_mean = 800` (epizod zawsze dobija do limitu kroków). Najlepsze polityki plateau-ują na 3/6, więc formalnie żaden epizod nie jest „pełnym sukcesem", choć połowa paczek trafia do bazy.

### 4.2 Wizualizacja zachowania (GIF z pierwszego epizodu)

In [ ]:
for label, run in RUNS_HOMO.items():
    p = GIFS_DIR / f'{run}.gif'
    if p.exists():
        display(Markdown(f'**{label}**'))
        display(Image(filename=str(p)))
    else:
        display(Markdown(f'_brak GIF-a `{p.name}`_'))

## 5. Trening heterogeniczny (różne algorytmy w jednym epizodzie)

Tutaj agenci stale współpracują w tym samym środowisku, ale każdy ma własną sieć i własny algorytm. Krzywe rolloutowe rejestrowane są **per learner-runda** (TB tag prefix to `hetero_.../roundK_robot_X_<algo>`).

In [ ]:
hetero_root = LOGS_DIR / RUN_HETERO
if hetero_root.exists():
    # Dla każdego agenta sklejamy krzywe ze wszystkich rund w jedną linię
    per_agent: dict[str, list[tuple[int, float]]] = {}
    # podkatalogi roundK_robot_X_algo mogą być zagnieżdżone (TensorBoard tb_log_name) -> rglob
    robot_dirs = sorted(p for p in hetero_root.rglob('*robot_*') if p.is_dir())
    for sub in robot_dirs:
        name = sub.name
        aid = 'robot_' + name.split('robot_')[1].split('_')[0]
        data = load_tb_scalars(sub, TAGS)
        if 'rollout/ep_rew_mean' in data:
            steps, vals = data['rollout/ep_rew_mean']
            per_agent.setdefault(aid, []).extend(zip(steps, vals))
    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    for aid, pts in sorted(per_agent.items()):
        pts.sort(key=lambda x: x[0])
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        ax.plot(xs, ys, label=aid, linewidth=1.4)
    ax.set_xlabel('timesteps (per agent, sklejone rundy)')
    ax.set_ylabel('mean episode reward (rollout)')
    ax.set_title('Krzywe uczenia — tryb heterogeniczny (round-robin)')
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(EVAL_DIR / 'learning_curves_hetero.png', dpi=120)
    plt.show()
else:
    display(Markdown(f'_Brak logów `{hetero_root}` — uruchom `train_multi_hetero.py`._'))

#### Co pokazują krzywe per-learner

Krzywe wyglądają **obiecująco w izolacji**: każdy z trzech agentów w trakcie swoich rund dochodzi do nagrody rzędu **+200** (max ~246 dla SAC w rundzie 1, ~238 dla PPO `robot_0` w rundzie 3). Trzeba je jednak czytać ostrożnie:

- To nagroda **pojedynczego uczącego się agenta**, podczas gdy **pozostali dwaj są zamrożeni** — czyli wynik „solisty grającego przeciw nieruchomym partnerom", a nie całej trójki działającej naraz. Nie jest to ta sama wielkość, co nagroda zespołowa z ewaluacji (sekcja 5.1).
- Trening jest **niestabilny między rundami**. 

| agent | runda 4: start → koniec | najlepsza runda (max) |
|---|---|---|
| robot_0 (PPO-v1) | +152 → +85 | ~238 (r3) |
| robot_1 (PPO-v2) | +85 → **−225** | ~233 (r3) |
| robot_2 (SAC) | +38 → +198 | ~246 (r1) |

### 5.1 Metryki końcowe trybu hetero


In [ ]:
p = EVAL_DIR / 'hetero.json'
if p.exists():
    agg = json.loads(p.read_text())['agg']
    display(pd.DataFrame([{'bieg': 'Hetero (PPO+PPO+SAC)', **agg}]).set_index('bieg').round(2))
else:
    display(Markdown('_Brak `hetero.json` — uruchom eval zgodnie z instrukcją powyżej._'))

p_gif = GIFS_DIR / 'hetero.gif'
if p_gif.exists():
    display(Image(filename=str(p_gif)))

#### Dlaczego wspólny eval wypada dużo gorzej niż krzywe

Indywidualne krzywe sięgają +200, a **wspólna ewaluacja całej trójki daje −1295.2** (3/6 dostaw, ale **292 kolizje** na epizod). Rozbieżność wynika z **niestacjonarności uczenia niezależnego**:

- Każdy agent uczył się przeciw **zamrożonym migawkom** pozostałych. Gdy złączymy trzy *finalne* polityki naraz, żadna nie widziała tej konkretnej kombinacji partnerów → przestają się skutecznie omijać.
- Porównanie z homo PPO jest wymowne: hetero dostarcza **tyle samo paczek (3/6)** co homo PPO, ale płaci **292 kolizjami zamiast 1**. Praktycznie cała strata to **brak koordynacji unikania kolizji** — dokładnie ten problem, który parameter sharing w trybie homo rozwiązuje implicite (jedna wspólna polityka → przewidywalne, spójne zachowanie), a round-robin niezależnych polityk nie.


## 6. Porównanie homo vs hetero

Zestawione metryki z wszystkich biegów:

In [ ]:
rows = []
for label, run in RUNS_HOMO.items():
    p = EVAL_DIR / f'{run}.json'
    if p.exists():
        rows.append({'bieg': f'Homo {label}', **json.loads(p.read_text())['agg']})
p = EVAL_DIR / 'hetero.json'
if p.exists():
    rows.append({'bieg': 'Hetero (PPO-v1 + PPO-v2 + SAC)', **json.loads(p.read_text())['agg']})
if rows:
    df = pd.DataFrame(rows).set_index('bieg').round(2)
    display(df)
    df.to_csv(EVAL_DIR / 'comparison.csv')
else:
    display(Markdown('_Brak danych eval — odpal `eval_multi.py` dla wszystkich biegów._'))

### 6.1 Dyskusja porównawcza

- **PPO ≫ SAC** w tym środowisku, w obu trybach. PPO (on-policy) stabilnie uczy się zbierać paczki; SAC (off-policy) przy tej skali kar zapada się w politykę bezruchu. Jest to kontrintuicyjne względem typowej tezy „SAC jest bardziej sample-efficient", ale pokazuje, że dobór algorytmu zależy od **ukształtowania nagrody**, a nie tylko od teoretycznej wydajności próbkowej.
- **PPO-v1 ≈ PPO-v2** — wariant hiperparametrów miał marginalny wpływ na wynik końcowy (różnica < 2 pkt nagrody, identyczne 3/6 dostaw i 1 kolizja).
- **Homo > Hetero** na metryce zespołowej, i to wyraźnie (−214…+164 vs −1295). Parameter sharing wygrywa **nie dlatego, że dostarcza więcej paczek** (tu remis 3/6 z PPO), lecz dlatego, że **eliminuje niestacjonarność** — wszystkie roboty współdzielą jedną, spójną politykę unikania kolizji (1 vs 292 kolizje). Hetero round-robin ma teoretyczny potencjał specjalizacji ról, ale w praktyce przegrywa przez brak ko-adaptacji finalnych polityk.

## 7. Wnioski

Eksperymenty objęły trzy biegi homogeniczne (PPO-v1, PPO-v2, SAC — parameter sharing) oraz jeden bieg heterogeniczny (PPO-v1 + PPO-v2 + SAC, round-robin self-play), każdy ewaluowany na 30 epizodach.

- **Uczenie zachodzi i jest widoczne na krzywych.** PPO w trybie homo poprawia nagrodę z ~−5900 do ~+165 i uczy się dostarczać 3/6 paczek przy ~1 kolizji. W trybie hetero każdy learner indywidualnie dochodzi do ~+200. Wymóg „umieść krzywą uczenia" jest spełniony dla wszystkich biegów.
- **Wybór algorytmu zależy od ukształtowania nagrody.** PPO radzi sobie dobrze; SAC przy obecnej skali kar kolapsuje do polityki bezruchu (0 dostaw) — to nasza główna negatywna obserwacja i naturalny kierunek na przyszłość: re-skalowanie / normalizacja nagrody, dłuższy `learning_starts`, reward shaping zachęcający do zbliżania się do paczek.
- **Wariant hiperparametrów PPO** (v1 vs v2) miał przy tym budżecie znikomy wpływ na wynik końcowy.
- **Homo vs hetero — kluczowy wniosek:** tryb z parameter sharing wyraźnie przewyższa heterogeniczny na metryce zespołowej. Hetero dostarcza tyle samo paczek, ale generuje ~292 kolizje/epizod zamiast 1 — cała różnica to **niestacjonarność uczenia niezależnego** (finalne polityki nie są ze sobą zgrane). To czytelna ilustracja centralnego problemu MARL.
- **Co poprawić w hetero:** (1) zapisywać najlepszą rundę per agent zamiast ostatniej (`robot_1` został zapisany po kolapsie rundy 4); (2) krótsze rundy / niższy LR, by ograniczyć przeuczenie przeciw zamrożonym partnerom; (3) docelowo zamienić round-robin na równoległe uczenie wszystkich (np. wspólny replay / centralised critic).
